# Climate Risk Assessment Framework – Google Colab

**What this notebook does:** Runs the Climate Risk Assessment Framework in the cloud. The framework estimates how climate risk affects the value (Net Asset Value, or NAV) of renewable energy assets (solar and wind)—this is called *NAV impairment analysis*.

**What you need:**
- A copy of `climate_risk_framework.tar.gz` on your Google Drive.
- This notebook opened in Google Colab (File → Upload notebook, or open from Drive with “Open with Colab”).

**How to use:** Run the cells **from top to bottom in order**. Do not skip cells.

**Important:** Colab sessions are temporary. Download any result files you need from the Colab file browser (left sidebar) **before** you close the tab or disconnect—otherwise they are lost.

---

**Before you start – quick checklist:**
- [ ] I have `climate_risk_framework.tar.gz` on Google Drive.
- [ ] I know the path to that file (e.g. `MyDrive/.../climate_risk_framework.tar.gz`).
- [ ] I will run cells in order and download outputs before closing.

## 1. Mount Google Drive and load the project

**What this section does:** Connects Colab to your Google Drive and loads the framework from a single archive file (`.tar.gz`). The project is extracted so the rest of the notebook can run it.

**What to change:** If your archive is in a different Drive folder, edit the `DRIVE_ARCHIVE_PATH` in the next cell to match your path (e.g. `"/content/drive/MyDrive/YourFolder/climate_risk_framework.tar.gz"`).

**What to expect:** You may be asked to allow Colab to access Drive (click “Connect to Google Drive” and approve). Then you should see `Project ready at: /content/climate_risk_framework`.

**If it fails:** An “Archive not found” error means the path is wrong or the file is not at that location in Drive. Double-check the path and that the file exists.

In [ ]:
import os

# Path to the framework archive on your Google Drive (after mount). Change this if your file is elsewhere.
DRIVE_ARCHIVE_PATH = "/content/drive/MyDrive/aamani/Modeling/Code/prashant/climate_risk_framework.tar.gz"

# Mount Google Drive (no-op if already mounted)
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("Drive already mounted.")

assert os.path.exists(DRIVE_ARCHIVE_PATH), f"Archive not found at {DRIVE_ARCHIVE_PATH}"

# Extract the archive to /content and set PROJECT_DIR (used by all later cells)
!cp "{DRIVE_ARCHIVE_PATH}" /content/
!tar -xzf /content/climate_risk_framework.tar.gz -C /content

PROJECT_DIR = "/content/climate_risk_framework"
assert os.path.exists(os.path.join(PROJECT_DIR, "main.py")), "main.py not found after extract"
print(f"Project ready at: {PROJECT_DIR}")

## 2. Install dependencies

**What this section does:** Installs the Python packages the framework needs (e.g. pandas, numpy, scipy, dash, plotly). These are listed in the project’s `requirements.txt`.

**What to expect:** The cell runs for a short time and finishes with a message like “Dependencies installed.” Run this once per Colab session (or again if you get “module not found” errors later).

In [ ]:
# Change into the project folder so requirements.txt is found, then install packages
%cd $PROJECT_DIR
!pip install -q -r requirements.txt 2>/dev/null || pip install -q pandas numpy scipy requests plotly dash matplotlib seaborn
print("Dependencies installed.")

## 3. Run the analysis

**What this section does:** Runs the full climate risk analysis pipeline: climate data acquisition, SCVR (Severe Climate Variability Rating), asset performance, business interruption, equipment degradation, and NAV (Net Asset Value) impairment. Results are written to the `outputs/` folder.

**What to choose (edit the next cell):**
- **ASSET:** `hayhurst_solar` (solar farm) or `maverick_wind` (wind farm).
- **SCENARIO:** `rcp45` (moderate emissions) or `rcp85` (high emissions).
- **RUN_MONTE_CARLO:** `True` to run uncertainty quantification (slower); `False` for a single deterministic run.

**What to expect:** The run uses `--quiet` so the cell doesn't print hundreds of log lines (Colab can stop runs when output is too large). Full logs are still written to `outputs/climate_risk_*.log`. The first run can take about 15–25 minutes. When it finishes, result files appear in `outputs/` (see Section 4).

**Tip:** You can re-run this cell with different `ASSET` or `SCENARIO` without re-running the earlier cells.

**If the run stops by itself (you didn't press Stop):** Colab sometimes kills the run when RAM spikes. The framework now frees memory between steps to reduce the spike. If it still stops, try **Runtime → Change runtime type → High RAM** and run again.

In [ ]:
%cd $PROJECT_DIR

# Which asset to analyze: hayhurst_solar (solar) or maverick_wind (wind)
ASSET = "hayhurst_solar"
# Climate scenario: rcp45 (moderate) or rcp85 (high emissions)
SCENARIO = "rcp45"
# Set True to run Monte Carlo uncertainty quantification (takes longer)
RUN_MONTE_CARLO = False

cmd = f"python main.py --asset {ASSET} --scenario {SCENARIO}"
if RUN_MONTE_CARLO:
    cmd += " --monte-carlo"
cmd += " --quiet"   # Less console output so Colab doesn't stop the run (full log still in outputs/*.log)
!{cmd}

## 4. View outputs

**What this section does:** Lists the result files produced by the analysis so you can confirm the run and choose what to download.

**Where they are:** Inside the project, in the `outputs/` folder. Typical files include: `*_scvr.csv` (climate variability ratings), `*_revenue.csv`, `*_bi_losses.csv`, `*_final_results.csv` (NAV summary), and optionally `*_monte_carlo.csv` if you ran with Monte Carlo.

**How to download:** In the left sidebar, open the folder tree: **content** → **climate_risk_framework** → **outputs**. Right-click a file → **Download**. Do this before you disconnect or close the tab, or the files will be lost.

In [ ]:
# List files in outputs/ so you can confirm the run and download what you need
%cd $PROJECT_DIR
!ls -la outputs/

## 5. Optional: View dashboard (inline)

**What this section does:** Loads your results from `outputs/` and shows a **dashboard directly in this cell**—summary metrics (NAV ratio, IUL/EUL) plus SCVR and revenue charts. No server or new tab; everything appears below.

**When to run:** Optional. Run only after Section 3 has completed successfully. Use the dropdowns to switch asset and scenario.

In [ ]:
# Inline dashboard: auto-detects outputs and shows results in this cell
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display, clear_output
import ipywidgets as widgets

# Try project outputs dir; fallback for Colab if PROJECT_DIR is elsewhere
def get_outputs_dir():
    for base in [Path(PROJECT_DIR), Path("/content/climate_risk_framework"), Path(".")]:
        d = base / "outputs"
        if d.exists():
            return d
    return Path(PROJECT_DIR) / "outputs"

OUTPUTS_DIR = get_outputs_dir()

def load_outputs():
    out = {}
    if not OUTPUTS_DIR.exists():
        return out
    for f in OUTPUTS_DIR.glob("*.csv"):
        try:
            out[f.stem] = pd.read_csv(f)
        except Exception:
            pass
    return out

# From CSV stems (e.g. hayhurst_solar_rcp45_scvr), infer (asset, scenario) pairs
def get_available_selections(results):
    pairs = set()
    for stem in results:
        if "_rcp45_" in stem:
            scenario = "rcp45"
            asset = stem.split("_rcp45_")[0]
            pairs.add((asset, scenario))
        if "_rcp85_" in stem:
            scenario = "rcp85"
            asset = stem.split("_rcp85_")[0]
            pairs.add((asset, scenario))
    return sorted(pairs)

def update_dashboard(asset_id, scenario, out):
    with out:
        clear_output(wait=True)
        results = load_outputs()
        final_key = f"{asset_id}_{scenario}_final_results"
        scvr_key = f"{asset_id}_{scenario}_scvr"
        rev_key = f"{asset_id}_{scenario}_revenue"

        if not results:
            print(f"No CSV files found in {OUTPUTS_DIR}. Run Section 3 first.")
            return

        # Summary
        if final_key in results and len(results[final_key]) > 0:
            row = results[final_key].iloc[0]
            nav = row.get("nav_impairment_ratio", 0)
            iul = row.get("iul_years", 0)
            eul = row.get("eul_years", 25)
            print(f"NAV Impairment Ratio: {nav:.3f}   |   IUL / EUL: {iul:.0f} / {eul:.0f} years")
        else:
            if scvr_key in results or rev_key in results:
                print("Summary (final_results not found); showing SCVR and Revenue below.")
            else:
                print("No results for this selection. Run Section 3 first.")

        # SCVR plot
        if scvr_key in results and not results[scvr_key].empty and "variable" in results[scvr_key].columns:
            df = results[scvr_key]
            fig_scvr = go.Figure()
            for var in df["variable"].unique():
                sub = df[df["variable"] == var]
                fig_scvr.add_trace(go.Scatter(x=sub["year"], y=sub["scvr"], mode="lines+markers", name=var))
            fig_scvr.update_layout(title="SCVR Over Time", xaxis_title="Year", yaxis_title="SCVR", template="plotly_white")
        else:
            fig_scvr = go.Figure().update_layout(title="SCVR Over Time (no data)")
        display(fig_scvr)

        # Revenue plot
        if rev_key in results and not results[rev_key].empty and "revenue_usd" in results[rev_key].columns:
            df = results[rev_key]
            fig_rev = go.Figure()
            fig_rev.add_trace(go.Scatter(x=df["year"], y=df["revenue_usd"], mode="lines", name="Revenue (USD)"))
            fig_rev.update_layout(title="Revenue Projections", xaxis_title="Year", yaxis_title="Revenue (USD)", template="plotly_white")
        else:
            fig_rev = go.Figure().update_layout(title="Revenue (no data)")
        display(fig_rev)

asset_dd = widgets.Dropdown(options=[("Hayhurst Solar", "hayhurst_solar"), ("Maverick Wind", "maverick_wind")], value="hayhurst_solar", description="Asset:")
scenario_dd = widgets.Dropdown(options=[("RCP 4.5", "rcp45"), ("RCP 8.5", "rcp85")], value="rcp45", description="Scenario:")
out = widgets.Output()

# Auto-detect: if we have CSVs, set dropdowns to first available (asset, scenario)
results = load_outputs()
available = get_available_selections(results)
if available:
    first_asset, first_scenario = available[0]
    asset_dd.value = first_asset
    scenario_dd.value = first_scenario

def on_change(change):
    update_dashboard(asset_dd.value, scenario_dd.value, out)

asset_dd.observe(on_change, names="value")
scenario_dd.observe(on_change, names="value")

display(widgets.HBox([asset_dd, scenario_dd]), out)
update_dashboard(asset_dd.value, scenario_dd.value, out)